# 05 — Emission Spectroscopy with the Equilibrium Chemistry Emulator

Adapted from the upstream ExoJAX tutorial *Emission Spectroscopy with
Equilibrium Chemistry* (Hajime Kawahara, v2.1) so the chemistry layer is
served by the **VULCAN FastChem emulator** instead of the live ExoGibbs
backend. The science is unchanged: a CO + H2 atmosphere with H2-H2 CIA,
forward-modelled into a high-resolution emission spectrum, then run through
a NUTS retrieval over `(T0, alpha, logg, RV, vsini, log_C_H, log_O_H, ...)`.

For the gradient gates that qualify the emulator for HMC/NUTS, see
`04_gradient_verification.ipynb`. For the side-by-side benchmark of the
emulator vs the classical (ExoGibbs) backend inside the *same* retrieval,
see `06_classical_vs_emulator_retrieval.ipynb`.


In [ ]:
# ExoJAX runs in float64 by default; the emulator is trained in float32 and
# JAX down-casts at the boundary, which matches training precision.
from jax import config
config.update("jax_enable_x64", True)


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

_STYLE = Path("science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print(f"MODEL       : {MODEL}")
print(f"BUNDLE_PATH : {BUNDLE_PATH}")


## 1. Loading a molecular database with `mdb`

ExoJAX has an API for molecular databases called `mdb` (and `adb` for atomic
databases). Below we use Carbon monoxide from ExoMol — `CO/12C-16O/Li2015`.
Set the spectral resolution to `R = 100,000`, with a wavenumber span fully
covering CO 22920–23000 Å.


In [ ]:
from exojax.utils.grids import wavenumber_grid

nu_grid, wav, resolution = wavenumber_grid(
    22920.0, 23000.0, 1500, unit="AA", xsmode="premodit",
)
print(f"R = {resolution:.0f}  N(nu) = {len(nu_grid)}")


In [ ]:
from exojax.database.exomol.api import MdbExomol

mdb = MdbExomol(".database/CO/12C-16O/Li2015", nurange=nu_grid)


## 2. Cross-section computation with `OpaPremodit`

ExoJAX's memory-saved opacity calculator. We auto-tune the temperature
range on `[500, 1500] K`, matching the temperature range of the retrieval
prior below.


In [ ]:
from exojax.opacity import OpaPremodit

opa = OpaPremodit(
    mdb=mdb,
    nu_grid=nu_grid,
    auto_trange=[500.0, 1500.0],
    dit_grid_resolution=1.0,
    allow_32bit=True,
)


In [ ]:
P = 1.0  # bar
T_lo, T_hi = 500.0, 1500.0
xsv_lo = opa.xsvector(T_lo, P)
xsv_hi = opa.xsvector(T_hi, P)


In [ ]:
fig = plt.figure(figsize=(11, 3.5))
plt.plot(nu_grid, xsv_hi, label=f"T = {T_hi:.0f} K", lw=0.9)
plt.plot(nu_grid, xsv_lo, label=f"T = {T_lo:.0f} K", lw=0.9, alpha=0.8)
plt.yscale("log")
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("CO cross section (cm$^2$)")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Atmospheric radiative transfer

`art = ArtEmisPure(...)` builds a pure-emission RT solver on a 60-level
pressure grid spanning 100 → 1e-5 bar (chosen one decade above the bundle's
training floor of 1e-6 bar so float32 quantization can never slip below
the validator's tolerance). We seed it with a power-law temperature
profile, `T(P) = T0 * (P / P_ref) ** alpha`, the same shape the retrieval
will sample.


In [ ]:
from exojax.rt import ArtEmisPure

# The pressure top must sit inside the bundle's training pressure union
# (the shipped FastChem bundle uses 1e-6 → ~200 bar). 1e-5 keeps every
# `art.pressure` level safely above the training floor under float32
# quantization (the validator's tolerance is ~1 nano-decade).
art = ArtEmisPure(
    nu_grid=nu_grid,
    pressure_btm=1.0e2,
    pressure_top=1.0e-5,
    nlayer=60,
    rtsolver="ibased",
    nstream=8,
)
art.change_temperature_range(500.0, 1500.0)
Tarr = art.powerlaw_temperature(1500.0, 0.1)


## 4. Plug in the VULCAN emulator as the chemistry backend

`make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")` returns a
JAX function that maps `(temperatures_k, pressures_bar, X/H dict)`
→ `(nz, n_species)` mixing-ratio table in ExoJAX's level convention
(level 0 at TOA).

We pin the abundance reference point to the training anchors in
`src/constants.py::SOLAR_ABUNDANCES`; using AAG21 ratios (or any other solar
compilation) here would feed the emulator a reference offset by a few
percent per element from the distribution it learned.


In [ ]:
from src.constants import SOLAR_ABUNDANCES
from src.models.standalone_inference import load_model, make_fastchem_vmr_fn

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")
print(f"chemistry      : {bundle.chemistry_type}")
print(f"model type     : {bundle.model_type}")
print(f"output species : {species_labels}")
print(f"global order   : {bundle.data_contract['global_static_feature_order']}")
print(f"fixed_globals  : {bundle.fixed_globals}")

# Single source of truth for the species indices used downstream — defining
# them once here prevents the kind of stale-rename NameError the previous
# version of this notebook had (`idx_CO_vulcan` was never defined).
IDX_CO = species_labels.index("CO")
IDX_H2 = species_labels.index("H2")
print(f"IDX_CO = {IDX_CO}  | IDX_H2 = {IDX_H2}")

# Training-anchored solar (He, C, O, N, S)/H reference point.
global_inputs_ref = {key: float(value) for key, value in SOLAR_ABUNDANCES.items()}
print(f"reference X/H  : {global_inputs_ref}")


## 5. JIT-compile the emulator on the chosen pressure grid

The first call traces and compiles the forward pass against the
`(art.pressure, Tarr, global_inputs)` shapes; subsequent calls (e.g. inside
NUTS leapfrog) reuse the compiled XLA graph.


In [ ]:
import jax
import jax.numpy as jnp

vmr_jit = jax.jit(vmr_fn)

# First call includes compilation time.
vmr_compiled = vmr_jit(Tarr, art.pressure, global_inputs_ref)
print(f"output shape : {vmr_compiled.shape}  dtype = {vmr_compiled.dtype}")

phot = int(jnp.argmin(jnp.abs(art.pressure - 0.1)))
print(f"VMR @ P = {float(art.pressure[phot]):.3f} bar")
print(f"  CO  = {float(vmr_compiled[phot, IDX_CO]):.3e}")
print(f"  H2  = {float(vmr_compiled[phot, IDX_H2]):.3e}")


## 6. Quick sanity check against ExoGibbs and live FastChem

Three backends (emulator / live FastChem / ExoGibbs), one profile, one X/H
point. This cross-checks that the bundle and reference frame are consistent
before entering the retrieval. The detailed comparison lives in
`02_chemistry_comparison.ipynb` — this notebook only needs the headline
number.


In [ ]:
from src.models.classical_reference import (
    build_exogibbs_element_vector,
    build_exogibbs_species_indices,
    chemsetup_matched_to_fastchem,
    mean_abs_log10_error,
    resolve_vulcan_source_root,
    run_fastchem_online,
)
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile
from exojax.utils.zsol import nsol

EXOGIBBS_ELEMENT_MODE = "fastchem_proxy"  # apples-to-apples with training
FASTCHEM_SOURCE_ROOT = resolve_vulcan_source_root(bundle.config, project_root=PROJECT_ROOT)
chem = chemsetup_matched_to_fastchem(FASTCHEM_SOURCE_ROOT)
EG_IDX = build_exogibbs_species_indices(chem, species_labels)
EG_OPTS = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

eg_element_vector = build_exogibbs_element_vector(
    chem, global_inputs_ref, solar_abundances=nsol(), mode=EXOGIBBS_ELEMENT_MODE,
)
vmr_exogibbs = np.asarray(
    equilibrium_profile(
        chem, np.asarray(Tarr), np.asarray(art.pressure),
        eg_element_vector, Pref=1.0, options=EG_OPTS,
    ).x[:, EG_IDX]
)
vmr_fastchem = run_fastchem_online(
    FASTCHEM_SOURCE_ROOT,
    np.asarray(art.pressure), np.asarray(Tarr),
    global_inputs_ref, species_labels, bundle.config,
)

print(
    f"mean |Δlog10 VMR|  ML-FC = {mean_abs_log10_error(vmr_compiled, vmr_fastchem):.4f}  "
    f"FC-EG = {mean_abs_log10_error(vmr_fastchem, vmr_exogibbs):.4f}  "
    f"ML-EG = {mean_abs_log10_error(vmr_compiled, vmr_exogibbs):.4f}"
)


In [ ]:
fig = plt.figure(figsize=(15, 5))
ax1 = fig.add_subplot(131)
ax1.plot(Tarr, art.pressure)
ax1.invert_yaxis()
ax1.set_yscale("log")
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("Pressure (bar)")
ax1.set_title("Power-law T(P)")

ax2 = fig.add_subplot(132)
ax2.plot(vmr_compiled[:, IDX_H2], art.pressure, label="Emulator", alpha=0.8)
ax2.plot(vmr_fastchem[:, IDX_H2], art.pressure, label="FastChem", alpha=0.8, ls=":")
ax2.plot(vmr_exogibbs[:, IDX_H2], art.pressure, label="ExoGibbs", alpha=0.8, ls="--")
ax2.invert_yaxis()
ax2.set_xlim(1e-10, 2)
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("VMR (H2)")
ax2.legend()

ax3 = fig.add_subplot(133)
ax3.plot(vmr_compiled[:, IDX_CO], art.pressure, label="Emulator", alpha=0.8)
ax3.plot(vmr_fastchem[:, IDX_CO], art.pressure, label="FastChem", alpha=0.8, ls=":")
ax3.plot(vmr_exogibbs[:, IDX_CO], art.pressure, label="ExoGibbs", alpha=0.8, ls="--")
ax3.invert_yaxis()
ax3.set_xscale("log")
ax3.set_yscale("log")
ax3.set_xlabel("VMR (CO)")
ax3.legend()
plt.tight_layout()
plt.show()


## 7. Mass mixing ratio (MMR) of CO

Convert the emulator's VMR (n_CO / n_total) to a mass mixing ratio
(m_CO / m_total) using the layerwise approximation `mmw ≈ 2.33` (an H2/He
atmosphere). Use a per-layer mmw built from the full 17-species table for
science-grade work — see `06_classical_vs_emulator_retrieval.ipynb`.


In [ ]:
from exojax.atm.atmconvert import vmr_to_mmr
from exojax.database.molinfo.mass import isotope_molmass

# Use the JIT-compiled emulator output here so the rest of the notebook
# matches the retrieval forward model below (which uses the same vmr_fn).
vmr_co = vmr_compiled[:, IDX_CO]
vmr_h2 = vmr_compiled[:, IDX_H2]

mean_molecular_weight = 2.33  # H2/He approximation
molmass = isotope_molmass("12C-16O")
mmr_profile_co = vmr_to_mmr(vmr_co, molmass, mean_molecular_weight)
mmr_profile_h2 = vmr_to_mmr(vmr_h2, isotope_molmass("1H2"), mean_molecular_weight)

fig = plt.figure(figsize=(5.5, 5.5))
ax = fig.add_subplot(111)
ax.plot(mmr_profile_co, art.pressure, label="CO")
ax.plot(mmr_profile_h2, art.pressure, ls="--", label="H2")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("MMR")
ax.set_ylabel("Pressure (bar)")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Surface gravity

Approximate Jupiter analogue: 1 R_J, 10 M_J.


In [ ]:
from exojax.utils.astrofunc import gravity_jupiter

gravity = gravity_jupiter(1.0, 10.0)
print(f"gravity = {gravity:.2f} cm/s^2")


## 9. Continuum opacity: H2-H2 CIA

`cdb` provides the collision-induced absorption tables.


In [ ]:
from exojax.database.contdb import CdbCIA
from exojax.opacity import OpaCIA

cdbH2H2 = CdbCIA(".database/H2-H2_2011.cia", nurange=nu_grid)
opacia = OpaCIA(cdb=cdbH2H2, nu_grid=nu_grid)


## 10. Build the optical depth and run RT


In [ ]:
xsmatrix = opa.xsmatrix(Tarr, art.pressure)
dtau_CO = art.opacity_profile_xs(xsmatrix, mmr_profile_co, mdb.molmass, gravity)


In [ ]:
logacia_matrix = opacia.logacia_matrix(Tarr)
dtaucia = art.opacity_profile_cia(
    logacia_matrix, Tarr, vmr_h2, vmr_h2, mean_molecular_weight, gravity,
)
dtau = dtau_CO + dtaucia


In [ ]:
F = art.run(dtau, Tarr)

fig = plt.figure(figsize=(11, 3))
plt.plot(nu_grid, F)
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("flux (erg/s/cm$^2$/cm$^{-1}$)")
plt.title("Raw emission spectrum (no rotation, no IP)")
plt.tight_layout()
plt.show()


## 11. Contribution function


In [ ]:
from exojax.plot.atmplot import plotcf

cf = plotcf(nu_grid, dtau, Tarr, art.pressure, art.dParr)
plt.show()


## 12. Spectral operators: rotational broadening, instrumental profile, RV shift


In [ ]:
from exojax.postproc.specop import SopRotation, SopInstProfile
from exojax.utils.instfunc import resolution_to_gaussian_std

vsini = 10.0  # km/s
u1, u2 = 0.0, 0.0
sop_rot = SopRotation(nu_grid, vsini_max=100.0)
Frot = sop_rot.rigid_rotation(F, vsini, u1, u2)

fig = plt.figure(figsize=(15, 4))
plt.plot(nu_grid, F, alpha=0.5, label="raw")
plt.plot(nu_grid, Frot, label="rotational broadening")
plt.legend()
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("flux")
plt.tight_layout()
plt.show()


In [ ]:
res_inst = 70000.0
beta_inst = resolution_to_gaussian_std(res_inst)
RV = 40.0  # km/s

sop_inst = SopInstProfile(nu_grid, vrmax=1000.0)
Finst = sop_inst.ipgauss(Frot, beta_inst)

# Instrument-binned wavelength grid.
nu_obs = nu_grid[::5][:-50]
mu = sop_inst.sampling(Finst, RV, nu_obs)

fig = plt.figure(figsize=(12, 6))
plt.plot(nu_grid, F, alpha=0.5, label="raw")
plt.plot(nu_grid, Frot, alpha=0.5, label="rot")
plt.plot(nu_grid, Finst, alpha=0.6, label="rot + IP")
plt.plot(nu_obs, mu, "o", ms=2, label="rot + IP + RV (obs)")
plt.legend()
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("flux")
plt.tight_layout()
plt.show()


## 13. Forward model `fspec(...)`

`fspec` packages the entire physics chain (chemistry → opacity → RT →
broadening) into one call. The retrieval and the mock-data generation use
the same function, only the inputs differ.

The chemistry layer here is `vmr_fn(...)` (the JAX-traceable emulator
callable). It replaces the ExoGibbs path used in the upstream tutorial,
which is what makes this notebook end-to-end differentiable through a
neural surrogate instead of through a Gibbs-minimization solve.


In [ ]:
def fspec(T0, alpha, g, RV, vsini, global_inputs):
    """Emission spectrum — emulator chemistry + ExoJAX RT."""
    Tarr = art.powerlaw_temperature(T0, alpha)
    xsmatrix = opa.xsmatrix(Tarr, art.pressure)

    # VMR profile from the emulator (replaces equilibrium_profile() in the
    # upstream notebook).
    vmr = vmr_fn(Tarr, art.pressure, global_inputs)
    vmr_co = vmr[:, IDX_CO]
    vmr_h2 = vmr[:, IDX_H2]

    mmr_co = vmr_to_mmr(vmr_co, molmass, mean_molecular_weight)
    dtau = art.opacity_profile_xs(xsmatrix, mmr_co, molmass, g)

    logacia = opacia.logacia_matrix(Tarr)
    dtau_cia = art.opacity_profile_cia(
        logacia, Tarr, vmr_h2, vmr_h2, mean_molecular_weight, g,
    )
    dtau = dtau + dtau_cia

    F = art.run(dtau, Tarr)
    Frot = sop_rot.rigid_rotation(F, vsini, u1, u2)
    Finst = sop_inst.ipgauss(Frot, beta_inst)
    return sop_inst.sampling(Finst, RV, nu_obs)


In [ ]:
fig = plt.figure(figsize=(12, 3))
plt.plot(nu_obs, fspec(1200.0, 0.09, gravity_jupiter(1.0, 1.0),  40.0, 10.0, global_inputs_ref), alpha=0.6, label="(1200, 0.09, 1 MJ)")
plt.plot(nu_obs, fspec(1100.0, 0.12, gravity_jupiter(1.0, 10.0), 20.0,  5.0, global_inputs_ref), alpha=0.6, label="(1100, 0.12, 10 MJ)")
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("flux")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 14. Mock observation

Generate a noiseless spectrum at a chosen truth, add Gaussian noise, and
keep that as the data the retrieval will fit.


In [ ]:
T0_truth, alpha_truth = 1295.0, 0.099
RV_truth, vsini_truth = 40.0, 10.0
g_truth = gravity_jupiter(1.0, 10.0)

mu_true = fspec(T0_truth, alpha_truth, g_truth, RV_truth, vsini_truth, global_inputs_ref)
NOISE = 500.0
rng = np.random.default_rng()
Fobs = np.asarray(mu_true) + rng.normal(0.0, NOISE, size=len(nu_obs))

fig = plt.figure(figsize=(12, 3.5))
plt.errorbar(nu_obs, Fobs, NOISE, fmt=".", color="gray", alpha=0.4, label="mock data")
plt.plot(nu_obs, mu_true, lw=1.0, label="truth (noiseless)")
plt.xlabel("wavenumber (cm$^{-1}$)")
plt.ylabel("flux (erg/s/cm$^2$/cm$^{-1}$)")
plt.legend()
plt.tight_layout()
plt.show()


## 15. Retrieval — NumPyro NUTS

A probabilistic model that varies six retrieval parameters (`T0`, `alpha`,
`logg`, `RV`, `vsini`, and `logZ`, an overall metallicity scale that
multiplies C/H and O/H) plus the Gaussian noise level. The five-element
emulator globals are built **inside** the model from the sampled
parameters, so HMC walks through real chemistry-dependent gradients on
every leapfrog step.


In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random


In [ ]:
def model_prob(spectrum):
    # Atmospheric / spectral parameters.
    logg = numpyro.sample("logg", dist.Uniform(4.0, 5.0))
    RV = numpyro.sample("RV", dist.Uniform(35.0, 45.0))
    T0 = numpyro.sample("T0", dist.Uniform(1100.0, 1500.0))
    alpha = numpyro.sample("alpha", dist.Uniform(0.05, 0.15))
    vsini = numpyro.sample("vsini", dist.Uniform(5.0, 15.0))
    logZ = numpyro.sample("logZ", dist.Uniform(-1.0, 1.0))  # log10 multiplier on C/H, O/H
    scale = 10.0 ** logZ

    # Build the emulator's global-inputs dict by scaling C/H and O/H by 10^logZ
    # (same prior choice as the original ExoGibbs version of this tutorial).
    global_inputs_in = {
        "He_H": global_inputs_ref["He_H"],
        "C_H":  global_inputs_ref["C_H"] * scale,
        "O_H":  global_inputs_ref["O_H"] * scale,
        "N_H":  global_inputs_ref["N_H"],
        "S_H":  global_inputs_ref["S_H"],
    }

    mu = fspec(T0, alpha, 10.0 ** logg, RV, vsini, global_inputs_in)
    sigmain = numpyro.sample("sigmain", dist.Exponential(1.0e-3))
    numpyro.sample("spectrum", dist.Normal(mu, sigmain), obs=spectrum)


NUTS settings. The forward model takes ~tens of milliseconds per call once
the JAX trace is cached, so 200 warmup + 300 samples typically completes in
under an hour on a CPU. Bump these up for production.


In [ ]:
rng_key = random.PRNGKey(int(np.random.default_rng().integers(0, 2 ** 31 - 1)))
rng_key, rng_key_run = random.split(rng_key)
NUM_WARMUP, NUM_SAMPLES = 200, 300

kernel = NUTS(model_prob, forward_mode_differentiation=False, max_tree_depth=11)


In [ ]:
mcmc = MCMC(kernel, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES)
mcmc.run(rng_key_run, spectrum=Fobs)
mcmc.print_summary()


## 16. Posterior predictive check


In [ ]:
from numpyro.diagnostics import hpdi
from numpyro.infer import Predictive
import jax.numpy as jnp

posterior_samples = mcmc.get_samples()
predictive = Predictive(model_prob, posterior_samples, return_sites=["spectrum"])
predictions = predictive(rng_key_run, spectrum=None)
median_mu = jnp.median(predictions["spectrum"], axis=0)
hpdi_band = hpdi(predictions["spectrum"], 0.9)


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(15, 4.5))
ax.plot(nu_obs, median_mu, color="C1", label="posterior median")
ax.fill_between(
    nu_obs, hpdi_band[0], hpdi_band[1],
    alpha=0.3, color="C1", label="90% HPDI",
)
ax.errorbar(nu_obs, Fobs, NOISE, fmt=".", label="mock spectrum", color="black", alpha=0.5)
ax.set_xlabel("wavenumber (cm$^{-1}$)")
ax.legend()
plt.tight_layout()
plt.show()


## 17. Corner plot


In [ ]:
import arviz

plt.style.use('science.mplstyle')

idata = arviz.from_numpyro(mcmc)
arviz.plot_pair(
    idata,
    kind="kde",
    divergences=False,
    marginals=True,
    var_names=['T0', 'alpha', 'logg', 'logZ', 'vsini', 'RV'],
)
plt.tight_layout()
plt.show()


The mass–metallicity degeneracy is the headline result of this notebook:
heavier planets (higher `logg`) and metal-richer atmospheres (higher
`logZ`) produce nearly the same emission spectrum at this resolution.
